In [5]:
import numpy as np

## Реализация K-Means

**Алгоритм:**
1. Инициализация $k$ центров случайным образом из данных
2. Повторение до сходимости:
   - Назначить каждую точку к ближайшему центру
   - Пересчитать центры как средние значения точек в каждом кластере

In [2]:
class KMeans:
    def __init__(self, n_clusters=3, max_iter=100, random_state=None):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.random_state = random_state
        self.centroids = None
        self.labels_ = None
    
    
    # Метод для обучения алгоритма кластеризации K-Means
    def fit(self, X):
        if self.random_state is not None:
            np.random.seed(self.random_state)
        
        # 1. Инициализация k случайных точек, как начальных центров
        # Выбор k случайных индексов
        random_indices = np.random.choice(a=X.shape[0], size=self.n_clusters, replace=False) 
        # Выбор k наблюдений как центров по индексам
        self.centroids = X[random_indices].copy()  
        
        # 2. Итерации алгоритма
        for _ in range(self.max_iter):
            # Отнесение каждого наблюдения к определенному кластеру (к кластеру с ближайшим центром)
            self.labels_ = self._assign_clusters(X)  
            
            # Нахождение новых центров кластеров с учетом найденных меток
            new_centroids = self._compute_centroids(X) 

            # Критерий останова -- если прошлые и текущие центры совпадают -- остановка
            if np.allclose(a=new_centroids, b=self.centroids):
                break
            
            # Сохранение новых значений центров кластеров в атрибут объекта  
            self.centroids = new_centroids
        return self
    
    
    # Метод назначения каждой точки к ее ближайшему центру
    def _assign_clusters(self, X):
        labels = np.array([np.linalg.norm(X - mu, axis=1, ord=2) for mu in self.centroids]).argmin(axis=0)
        return labels
    
    
    # Метод вычисления новых центров кластеров, как среднего точек в кластере
    def _compute_centroids(self, X):
        centroids = [X[self.labels_ == k].mean(axis=0) for k in range(self.n_clusters)]
        return np.array(centroids)
    
    
    # Метод предсказания кластеров для новых данных
    def predict(self, X):
        return self._assign_clusters(X)

---

## Реализация K-Means++

**Алгоритм инициализации K-Means++:**
1. Выбор первого центра случайно
2. Для каждого следующего центра:
   - Для каждой точки вычислить расстояние до ближайшего уже выбранного центра
   - Выбрать следующий центр с вероятностью пропорциональной квадрату расстояния

In [4]:
class KMeansPlusPlus:
    def __init__(self, n_clusters=3, max_iter=100, random_state=None):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.random_state = random_state
        self.centroids = None
        self.labels_ = None
    
    
    # Метод для обучения алгоритма кластеризации K-Means++
    def fit(self, X):
        if self.random_state is not None:
            np.random.seed(self.random_state)
        
        # 1. Инициализация K-Means++
        self.centroids = self._init_centroids_plus_plus(X)
        
        # 2. Итерации алгоритма
        for _ in range(self.max_iter):
            # Отнесение каждого наблюдения к определенному кластеру (к кластеру с ближайшим центром)
            self.labels_ = self._assign_clusters(X)  
            
            # Нахождение новых центров кластеров с учетом найденных меток
            new_centroids = self._compute_centroids(X) 

            # Критерий останова -- если прошлые и текущие центры совпадают -- остановка
            if np.allclose(a=new_centroids, b=self.centroids):
                break
            
            # Сохранение новых значений центров кластеров в атрибут объекта  
            self.centroids = new_centroids
        return self
    
    # Метод для инициализации центров кластеров методом K-Means++
    def _init_centroids_plus_plus(self, X):
        
        # Шаг 1 - Выбор первого центра кластера случайно и добавление его в массив для хранения
        first_idx = np.random.choice(a=X.shape[0], size=1) 
        centroids = [X[first_idx]]
        
        # Шаг 2 - Выбер остальных k - 1 центров кластеров 
        for _ in range(self.n_clusters - 1):
            # Преобразование в массив NumPy для векторной работы
            centroids_arr = np.atleast_2d(centroids)
             
            # Нахождение квадрата расстояния до ближайшего центра для каждой точки 
            distances = np.array([
                np.linalg.norm(X - c, ord=2, axis=1)
                for c in centroids_arr
            ]).min(axis=0)**2
            
            # Создание массива вероятности выбора точек следующим центром (пропорционально квадрату расстояния)
            probabilities = distances / distances.sum()
            
            # Выбор следующего центра кластера, согласно вероятностям
            # Точки которые уже являются центрами будут иметь p = 0 так как их расстояние до ближ. центра 0
            next_idx = np.random.choice(a=X.shape[0], size=1, replace=False, p=probabilities)
            
            # Добавление найденного центра кластера в массив
            centroids.append(X[next_idx])
        return np.array(centroids)
    
    
     # Метод для назначения каждой точки к ей ближайшему центру
    def _assign_clusters(self, X):
        labels = np.array([np.linalg.norm(X - mu, axis=1, ord=2) for mu in self.centroids]).argmin(axis=0)
        return labels
    
    
    # Метод для вычисления новых центров кластеров, как среднего точек в кластере
    def _compute_centroids(self, X):
        centroids = [X[self.labels_ == k].mean(axis=0) for k in range(self.n_clusters)]
        return np.array(centroids)
    
    
    # Метод предсказания кластеров для новых данных
    def predict(self, X):
        return self._assign_clusters(X)